# A tour of the rebuilt ProbPipe API

This notebook tours the API on `dev/overhaul`, with one hierarchical model as the thread: the non-centered eight-schools model, whose parameters form the nested record `{population: {mu, tau}, groups: {theta_tilde}}`.
Each markdown cell says what the next cell shows, and each output is a library repr, declaration, report, or error message, or a table built from them.
A cell marked **Bug** shows a behavior that departs from the design reference in `design/` and names the section it departs from.
The last section lists what felt awkward while writing the notebook.

## Status

The pieces below are transitional, and a cell that meets one marks it **Transitional**.

1. **Imports.** The operations are imported from their modules until the package exports switch to them. Until then, `probpipe.sample`, `probpipe.mean`, and the other top-level names are the earlier implementations, which take `key=` arguments.
2. **Labels.** A term's label is read from `name` and set by `with_name` until the export switch renames the two `label` and `with_label`.
3. **`expectation`** resolves through its own method registry, whose methods are `exact` and `monte_carlo`, until it becomes `mean(evaluate(f, d))` and takes `evaluate`'s controls and method names.
4. **`cov`** returns a dense array for now, and a later change returns a linear operator over the flattened draw.
5. **`evaluate`** registers three rules for now: the empirical enumeration, the elementwise sweep, and the sampling lift. Its exact rules for closed forms, changes of variables, and linear maps come in later changes, so `evaluate` of a parametric law takes the sampling lift.
6. **Record arithmetic.** `+`, `-`, and scalar `*` and `/` on a `NumericRecord` raise `TypeError` until the vector-space operators are implemented, and `map` and `to_vector` serve until then.
7. **Monte Carlo marginals.** `marginal` has no Monte Carlo route until the empirical marginal of projected draws is implemented, so a field with no exact marginal raises `ResolutionError`.
8. **Docs.** The other notebooks in `docs/` use the earlier implementations until a later pass rewrites them.

## Setup

`show` prints several properties of one object as an aligned block, and `spec_table` tabulates the leaves of a declaration.
A `CallReport` displays as a table with one row per route and the selected route marked.

In [1]:
import tempfile
import warnings
from pathlib import Path
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from IPython.display import display

import probpipe
from probpipe import (
    BijectorTransformedDistribution, BootstrapDistribution, BootstrapReplicateDistribution,
    Dirichlet, DistributionBatch, EmpiricalDistribution, Gamma, HalfCauchy, KDEDistribution,
    MathematicalDomainError, MultivariateNormal, Normal, NumericArray, NumericArrayBatch,
    NumericArraySpec, NumericRecordBatch, Opaque, OutputSpec, Poisson, PyMCModel, Record,
    RecordBatch, RecordSpec, ResolutionError, ResultSchemaError, StanModel, StudentT,
    SupportsCovariance, SupportsLogProb, SupportsMean, SupportsQuantile, SupportsSampling,
    SupportsUnnormalizedLogProb, SupportsVariance, bijector_for, conditional_distribution,
    converter_registry, function, iterate, positive, provenance_ancestors, real, replay_run,
    workflow_run,
)
from probpipe.distributions import (
    ConditionalDistribution, SupportsConditionalLogProb, SupportsConditionalSampling,
    SupportsFactors, SupportsMarginals,
)
from probpipe.families import PoissonFamily, glm_likelihood
from probpipe.operations import operation_registry

# The operations, from their modules until the top-level exports switch.
from probpipe.operations._condition import condition_on
from probpipe.operations._convert import convert
from probpipe.operations._density import log_prob, unnormalized_log_prob
from probpipe.operations._evaluate import evaluate
from probpipe.operations._marginal import factor, marginal
from probpipe.operations._moments import cov, expectation, mean, quantile, variance
from probpipe.operations._sample import sample

warnings.filterwarnings("ignore", message=r"Explicitly requested dtype float64")
warnings.filterwarnings("ignore", message=r"Loading a shared object")


def show(properties):
    """Print each property on a line of its own, with the names aligned in one column."""
    width = max(map(len, properties))
    for name, value in properties.items():
        text = str(value).replace("\n", "\n" + " " * (width + 2))
        print(f"{name:<{width}}  {text}")


def spec_table(specs):
    """The shape, dtype, and support of each leaf of *specs*, a record spec or a mapping of specs."""
    leaves = {path: specs[path] for path in specs}
    return pd.DataFrame(
        {"shape": [spec.shape for spec in leaves.values()],
         "dtype": [spec.dtype for spec in leaves.values()],
         "support": [spec.support for spec in leaves.values()]},
        index=list(leaves),
    )


def show_report(report):
    display(report)

## 1. Values

A value is a tracked term: a representation together with a label and a spec.
`NumericArray` holds one array, whose shape is the event shape, and arithmetic on it returns a tracked term labeled by the expression.

In [2]:
effect = NumericArray("effect", jnp.array([28.0, 8.0, -3.0]))
show({"effect": repr(effect), "effect.spec": effect.spec,
      "effect + 1.0": repr(effect + 1.0), "2 * effect": repr(2 * effect)})

effect        NumericArray('effect', shape=(3,), dtype=float32)
effect.spec   NumericArraySpec(shape=(3,), dtype=float32)
effect + 1.0  NumericArray('effect + 1.0', shape=(3,), dtype=float32)
2 * effect    NumericArray('2 * effect', shape=(3,), dtype=float32)


A `Record` is a named tree of values, and a mapping-valued field becomes a subtree.
A subtree whose leaves are all numeric is promoted to a `NumericRecord`, so a school's record holds a numeric `data` subtree beside a string label.

In [3]:
school = Record("school_A", {"data": {"effect": 28.0, "se": 15.0}, "label": "A"})
show({"school": repr(school), "school.spec": school.spec,
      'school.at_path("data")': repr(school.at_path("data"))})

school                  Record('school_A', fields=('data/effect', 'data/se', 'label'))
school.spec             RecordSpec(data=NumericRecordSpec(effect=(), se=()), label=OpaqueSpec(type=str))
school.at_path("data")  NumericRecord('data', fields=('effect', 'se'))


Indexing a record by a key returns a tracked view of the field's kind, and the view is labeled by the key.

In [4]:
show({'school["data/effect"]': repr(school["data/effect"]), 'school["label"]': repr(school["label"])})

school["data/effect"]  NumericArray('data/effect', shape=(), dtype=float32)
school["label"]        Opaque('label', type=str)


The eight-schools parameters form a `NumericRecord`.
It has the flat-vector interface, and `map` applies an elementwise function and keeps the structure.

In [5]:
theta = Record("theta", {"population": {"mu": 4.0, "tau": 3.0},
                         "groups": {"theta_tilde": jnp.linspace(-1.0, 1.0, 8)}})
show({"theta": repr(theta), "theta.vector_size": theta.vector_size,
      "theta.to_vector()": np.asarray(theta.to_vector()).round(2),
      "theta.map(jnp.abs)": repr(theta.map(jnp.abs))})

theta               NumericRecord('theta', fields=('population/mu', 'population/tau', 'groups/theta_tilde'))
theta.vector_size   10
theta.to_vector()   [ 4.    3.   -1.   -0.71 -0.43 -0.14  0.14  0.43  0.71  1.  ]
theta.map(jnp.abs)  NumericRecord('theta', fields=('population/mu', 'population/tau', 'groups/theta_tilde'))


**Transitional:** record arithmetic raises `TypeError` (Status).

In [6]:
try:
    theta + theta
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

TypeError: unsupported operand type(s) for +: 'NumericRecord' and 'NumericRecord'


`Opaque` wraps a value that no other kind admits, such as a tuple, and its spec records the value's type.

In [7]:
note = Opaque("source", ("Rubin (1981)", "SAT points"))
show({"note": repr(note), "note.spec": note.spec, "note.value": note.value})

note        Opaque('source', type=tuple)
note.spec   OpaqueSpec(type=tuple)
note.value  ('Rubin (1981)', 'SAT points')


A mapping is refused, since the value layer reads a mapping as a subtree.

In [8]:
try:
    Opaque("source", {"study": "Rubin (1981)"})
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

TypeError: Opaque holds one unstructured value, and the value layer reads a mapping as a subtree rather than a leaf; wrap it as a Record, or as a non-mapping value


A `RecordBatch` stores records column by column on named levels.
A position gives a record.

In [9]:
y_obs = jnp.array([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0])
sigma = jnp.array([15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])
J = y_obs.shape[0]
schools_data = RecordBatch.stack(
    [Record("school", {"effect": y, "se": s, "label": label})
     for y, s, label in zip(y_obs, sigma, "ABCDEFGH")],
    level_name="school",
    name="schools",
)
show({"schools_data": repr(schools_data), "schools_data.element_spec": schools_data.element_spec,
      "schools_data[2]": repr(schools_data[2]),
      "schools_data.at_levels(school=2)": repr(schools_data.at_levels(school=2))})

schools_data                      RecordBatch('schools', levels={'school': 8}, fields=('effect', 'se', 'label'))
schools_data.element_spec         RecordSpec(effect=(), se=(), label=OpaqueSpec(type=str))
schools_data[2]                   Record('schools[school=2]', fields=('effect', 'se', 'label'))
schools_data.at_levels(school=2)  Record('schools[school=2]', fields=('effect', 'se', 'label'))


A field gives its column as a batch of the field's kind: an `OpaqueBatch` for the labels and a `NumericArrayBatch` for the effects.

In [10]:
show({'schools_data["label"]': repr(schools_data["label"]),
      'schools_data["effect"]': repr(schools_data["effect"])})

schools_data["label"]   OpaqueBatch("schools['label']", levels={'school': 8}, element_spec=OpaqueSpec(type=str))
schools_data["effect"]  NumericArrayBatch(
                            "schools['effect']",
                            levels={'school': 8},
                            element_spec=NumericArraySpec(shape=()),
                        )


Every kind defines `raw()`, which returns its representation detached from the workflow.
A value's raw form is its array, its nested mapping, or its stored object.
A parametric law's raw form is its backend object, and an empirical law's is the law itself, detached.

In [11]:
terms = [effect, school, note, schools_data, schools_data["effect"],
         EmpiricalDistribution("effect", y_obs), Normal("x", 0.0, 1.0)]
raw_forms = [term.raw() for term in terms]
pd.DataFrame({"type of raw()": [type(raw).__name__ for raw in raw_forms],
              "package": [type(raw).__module__.partition(".")[0] for raw in raw_forms]},
             index=[type(term).__name__ for term in terms])

,type of raw(),package
NumericArray,ArrayImpl,jaxlib
Record,dict,builtins
Opaque,tuple,builtins
RecordBatch,dict,builtins
NumericArrayBatch,ArrayImpl,jaxlib
EmpiricalDistribution,EmpiricalDistribution,probpipe
Normal,Normal,tensorflow_probability


## 2. Naming

A label names an object and has no mathematical meaning, while a component name addresses a part of what the object produces.
A family's component defaults to its label, and `event_spec` names another.

In [12]:
prior_beta = Normal("prior", 0.0, 1.0, event_spec=OutputSpec(beta=None))
prior_beta

Normal(
    'prior',
    loc=0.0,
    scale=1.0,
    event_spec=OutputSpec(beta=NumericArraySpec(shape=(), dtype=float32, support=real)),
)

`with_name` changes the label alone, and indexing a whole-term law by its component returns the law itself.

In [13]:
renamed = prior_beta.with_name("slope_prior")
show({"label": prior_beta.name, "components": list(prior_beta.event_spec.components),
      "label after with_name": renamed.name,
      "components after with_name": list(renamed.event_spec.components),
      'prior_beta["beta"] is prior_beta': prior_beta["beta"] is prior_beta})

label                             prior
components                        ['beta']
label after with_name             slope_prior
components after with_name        ['beta']
prior_beta["beta"] is prior_beta  True


`with_path_names` renames nodes by their exact paths.
A target under the same parent renames in place, and a target under another parent moves the node, which promotes or demotes it.
All renames apply at once, so a swap is one call.

In [14]:
swapped = school.with_path_names({"data/effect": "data/se", "data/se": "data/effect"})
show({"in place": list(school.with_path_names({"data/effect": "data/y"}).keys()),
      "promotion": list(school.with_path_names({"data/effect": "effect"}).keys()),
      "demotion": list(school.with_path_names({"label": "meta/label"}).keys()),
      "swap": f'data/effect = {swapped["data/effect"]}, data/se = {swapped["data/se"]}'})

in place   ['data/y', 'data/se', 'label']
promotion  ['data/se', 'label', 'effect']
demotion   ['data/effect', 'data/se', 'meta/label']
swap       data/effect = 15.0, data/se = 28.0


A key must be the exact path of a node, so the bare `effect` is refused, and the `KeyError` lists the paths.

In [15]:
try:
    school.with_path_names({"effect": "y"})
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

KeyError: "with_path_names(): 'effect' is not the path of a node; the paths are ['data', 'data/effect', 'data/se', 'label']"


The same call renames a law's event.
Demoting the fields of a joint of three parametric priors builds the eight-schools prior.
A rename that gathers the components of several factors under one node regroups those factors into a packaged sub-joint, so the renamed prior stays factored.

In [16]:
flat_prior = Normal("mu", 0.0, 5.0) * HalfCauchy("tau", 0.0, 5.0) * Normal("theta_tilde", jnp.zeros(J), 1.0)
NEST = {"mu": "population/mu", "tau": "population/tau", "theta_tilde": "groups/theta_tilde"}
prior = flat_prior.with_path_names(NEST)
prior

FactoredDistribution(
    'mu·tau·theta_tilde',
    factors=(
        FactoredDistribution(
            'mu·tau',
            factors=(Normal('mu', loc=0.0, scale=5.0), HalfCauchy('tau', loc=0.0, scale=5.0)),
            event_spec=OutputSpec(
                population=NumericRecordSpec(
                    mu=NumericArraySpec(shape=(), dtype=float32, support=real),
                    tau=NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0)),
                ),
            ),
        ),
        FactoredDistribution(
            'theta_tilde',
            factors=(
                Normal('theta_tilde', loc=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], scale=1.0),
            ),
            event_spec=OutputSpec(
                groups=NumericRecordSpec(
                    theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
                ),
            ),
        ),
    ),
)

Each gathering node is one factor, so the renamed prior has two factors and keeps `SupportsFactors`.

In [17]:
joints = {"flat_prior": flat_prior, "prior": prior}
pd.DataFrame({"components": [list(d.event_spec.components) for d in joints.values()],
              "factors": [[part.name for part in d.factors] for d in joints.values()],
              "SupportsFactors": [isinstance(d, SupportsFactors) for d in joints.values()]},
             index=list(joints))

,components,factors,SupportsFactors
flat_prior,"[mu, tau, theta_tilde]","[mu, tau, theta_tilde]",True
prior,"[population, groups]","[mu·tau, theta_tilde]",True


Demoting the component of a whole-term law is refused, since it would change the draw's kind.

In [18]:
try:
    Normal("mu", 0.0, 5.0).with_path_names({"mu": "population/mu"})
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

ValueError: with_path_names() keeps the packaging, so the whole term's component 'mu' is renamed in place, not moved to 'population/mu'


## 3. Distributions

Each family derives its event declaration from its parameters: the component, shape, dtype, and support.
Capabilities are protocols, so `isinstance(d, SupportsMean)` reports whether a law implements a mean.

In [19]:
families = [
    Normal("mu", 0.0, 5.0),
    HalfCauchy("tau", 0.0, 5.0),
    Poisson("count", 3.0),
    Dirichlet("weights", jnp.ones(3)),
    MultivariateNormal("z", jnp.zeros(2), cov=jnp.array([[1.0, 0.5], [0.5, 2.0]])),
]
protocols = [SupportsSampling, SupportsLogProb, SupportsMean, SupportsVariance, SupportsCovariance,
             SupportsQuantile, SupportsMarginals]
pd.DataFrame(
    {
        "component": [next(iter(d.event_spec.components)) for d in families],
        "shape": [d.event_shape for d in families],
        "dtype": [str(d.dtype) for d in families],
        "support": [str(d.support) for d in families],
        **{p.__name__.removeprefix("Supports"): [isinstance(d, p) for d in families] for p in protocols},
    },
    index=[type(d).__name__ for d in families],
)

,component,shape,dtype,support,Sampling,LogProb,Mean,Variance,Covariance,Quantile,Marginals
Normal,mu,(),float32,real,True,True,True,True,True,True,False
HalfCauchy,tau,(),float32,greater_than(0.0),True,True,True,True,True,True,False
Poisson,count,(),float32,non_negative_integer,True,True,True,True,True,False,False
Dirichlet,weights,"(3,)",float32,simplex,True,True,True,True,True,False,False
MultivariateNormal,z,"(2,)",float32,real,True,True,True,True,True,True,False


A guard narrows a claim per instance.
`StudentT` claims `SupportsMean`, and its guard admits the call once the degrees of freedom are concrete.
A mean that is undefined raises `MathematicalDomainError`, as a Student t's does at one degree of freedom or fewer, and a mean that diverges is `inf`, as the half-Cauchy's is.

In [20]:
t = StudentT("t", 0.8, 0.0, 1.0)
print("t claims SupportsMean:", isinstance(t, SupportsMean))
try:
    mean(t)
except MathematicalDomainError as error:
    print(f"{type(error).__name__}: {error}")
print("half-Cauchy mean:", mean(HalfCauchy("tau", 0.0, 5.0)))

t claims SupportsMean: True
MathematicalDomainError: the mean of 't' does not exist: it is defined only for degrees of freedom above one
half-Cauchy mean: inf


Parameters with more axes than one law needs give one law whose extra axes are event axes.
A vector `loc` gives independent coordinates, and a `MultivariateNormal` whose `loc` has shape `(4, 2)` gives four independent rows, so `log_prob` of one draw is a scalar in both cases.
Separate laws form a `DistributionBatch` (Section 8).

In [21]:
coords = Normal("theta_tilde", jnp.zeros(J), 1.0)
rows = MultivariateNormal("rows", jnp.zeros((4, 2)), cov=jnp.array([[1.0, 0.8], [0.8, 1.0]]))
with workflow_run(seed=0):
    draw = sample(rows)
show({"coords event": coords.event_spec.spec, "rows event": rows.event_spec.spec,
      "log_prob(coords, 0)": repr(log_prob(coords, jnp.zeros(J))),
      "log_prob(rows, draw)": repr(log_prob(rows, draw))})

coords event          NumericArraySpec(shape=(8,), dtype=float32, support=real)
rows event            NumericArraySpec(shape=(4, 2), dtype=float32, support=real)
log_prob(coords, 0)   NumericArray('theta_tilde', shape=(), dtype=float32)
log_prob(rows, draw)  NumericArray('rows', shape=(), dtype=float32)


## 4. Operations on laws

The operations form a registry, which lists each operation's operands and routes.

In [22]:
summaries = operation_registry.list()
pd.DataFrame(
    {"operands": [", ".join(operand.name for operand in s.operands) for s in summaries],
     "kind": ["derived" if s.is_derived else "primitive" for s in summaries],
     "routes": [", ".join(route.name for route in s.routes) for s in summaries]},
    index=[s.name for s in summaries],
)

,operands,kind,routes
convert,"d, target",primitive,"identity, converters, normalize"
sample,"d, sample_shape",primitive,"exact, normalize"
condition_on,"d, given",primitive,"curry, slice, exact_conditioning, bayes, appro..."
log_prob,"d, value",primitive,exact
unnormalized_log_prob,"d, value",primitive,exact
prob,"d, value",derived,identity
unnormalized_prob,"d, value",derived,identity
random_log_prob,M,primitive,exact
random_unnormalized_log_prob,M,primitive,exact
evaluate,"f, v, fixed_args",primitive,evaluation_rules


`describe` states one operation's contract, with its routes in selection order.

In [23]:
print(operation_registry.describe("mean"))

mean(d) — primitive
  The mean ``E[X]`` for ``X ~ d``, a value shaped like one draw.
  operands:
    d: DistributionSpec; read by the result rule
  routes, in selection order:
    closed_form (capability, exact; requires SupportsMean): membership in SupportsMean suffices
    monte_carlo (fallback, approximate): The law samples numeric draws, which average coordinatewise, or laws, which average to a mixture.


`sample` returns one tracked draw, or a batch on a level named `sample`, and each draw is labeled by the law it comes from.
`with_options(raw=True)` returns the representation instead, and draws inside `workflow_run(seed=...)` are reproducible (Section 12).

In [24]:
mu_prior = Normal("mu", 0.0, 5.0)
with workflow_run(seed=0):
    one = sample(mu_prior)
    many = sample(mu_prior, sample_shape=(5,))
    raw_many = sample.with_options(raw=True)(mu_prior, sample_shape=(5,))
show({"one draw": repr(one), "five draws": repr(many), "values of the five draws": many.values,
      "type of the raw draws": type(raw_many).__name__, "raw draws": raw_many})

one draw                  NumericArray('mu', shape=(), dtype=float32, support=real)
five draws                NumericArrayBatch(
                              'mu',
                              levels={'sample': 5},
                              element_spec=NumericArraySpec(shape=(), dtype=float32, support=real),
                          )
values of the five draws  [-5.127493  -1.9288195 -0.6630897  6.3674555  3.6620731]
type of the raw draws     ArrayImpl
raw draws                 [-14.860659   -3.721889   -6.036863    3.994918    6.0235777]


A joint's draws form a record batch whose fields follow the prior's nested record, and an element is labeled by its position.

In [25]:
with workflow_run(seed=0):
    prior_draws = sample(prior, sample_shape=(3,))
show({"prior_draws": repr(prior_draws), "prior_draws[0]": repr(prior_draws[0])})

prior_draws     NumericRecordBatch(
                    'mu·tau·theta_tilde',
                    levels={'sample': 3},
                    fields=('population/mu', 'population/tau', 'groups/theta_tilde'),
                )
prior_draws[0]  NumericRecord(
                    'mu·tau·theta_tilde[sample=0]',
                    fields=('population/mu', 'population/tau', 'groups/theta_tilde'),
                )


The functionals return values of the event's kind, labeled by the law.
**Transitional:** `cov` returns a dense array (Status).

In [26]:
z = families[-1]
show({"log_prob(z, [0, 0])": repr(log_prob(z, jnp.zeros(2))), "mean(z)": repr(mean(z)),
      "variance(z)": repr(variance(z)), "cov(z)": repr(cov(z)),
      "quantile(z, 0.5)": repr(quantile(z, 0.5))})

log_prob(z, [0, 0])  NumericArray('z', shape=(), dtype=float32)
mean(z)              NumericArray('z', shape=(2,), dtype=float32, support=real)
variance(z)          NumericArray('z', shape=(2,), dtype=float32, support=non_negative)
cov(z)               NumericArray('z', shape=(2, 2), dtype=float32)
quantile(z, 0.5)     NumericArray('z', shape=(2,), dtype=float32)


`quantile` at several levels returns a batch on a level named `quantile`, with the level axis leading, so its values have one row per level.

In [27]:
levels = quantile(z, jnp.array([0.05, 0.5, 0.95]))
show({"levels": repr(levels)})
pd.DataFrame(np.asarray(levels.values), columns=["z[0]", "z[1]"],
             index=pd.Index([0.05, 0.5, 0.95], name="quantile")).round(3)

levels  NumericArrayBatch(
            'z',
            levels={'quantile': 3},
            element_spec=NumericArraySpec(shape=(2,), dtype=float32),
        )


,z[0],z[1]
quantile,,
0.05,-1.645,-2.326
0.50,0.000,0.000
0.95,1.645,2.326


`expectation(d, f)` integrates `f`, here by Monte Carlo over 128 draws.
**Transitional:** `expectation` resolves through its own method registry (Status).

In [28]:
@function
def square(x):
    return x**2


with workflow_run(seed=0):
    second_moment = expectation(mu_prior, square)
show({"expectation(mu_prior, square)": float(second_moment), "exact value": 25.0})

expectation(mu_prior, square)  23.68588638305664
exact value                    25.0


`evaluate(f, d)` returns the pushforward law, which for a nonlinear map is an empirical law over the outputs, labeled by the map.

In [29]:
with workflow_run(seed=0):
    pushforward = evaluate(square, mu_prior)
pushforward

EmpiricalDistribution(
    'square',
    atoms=NumericArrayBatch(
        'square',
        levels={'draw': 128},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
)

Its provenance records the rule that realized it, and its mean is the second moment again.

In [30]:
show({"pushforward.provenance.metadata": pushforward.provenance.metadata,
      "mean(pushforward)": float(mean(pushforward))})

pushforward.provenance.metadata  {'func': 'evaluate', 'route': 'evaluation_rules', 'exact': False, 'method': 'sampling_lift'}
mean(pushforward)                23.68588638305664


`check` runs the call up to the selection of a route without executing it, and returns a `CallReport`.
A report displays as a table with one row per route, and the selected route is marked.

In [31]:
mean.check(z)

,route,method,exact,feasible,reason
selected,closed_form,,exact,feasible,


The report's repr reads as the call of its constructor.

In [32]:
print(repr(mean.check(z)))

CallReport(
    routes=(MethodInfo(True, method_name='closed_form', exact=True),),
    selected=MethodInfo(True, method_name='closed_form', exact=True),
    result=OutputSpec(z=NumericArraySpec(shape=(2,), dtype=float32, support=real)),
)


`expectation`'s report shows its own registry: the exact method is infeasible for a `Normal`, so the Monte Carlo method is selected.

In [33]:
expectation.check(mu_prior, square)

,route,method,exact,feasible,reason
,methods,exact methods,exact,infeasible,No feasible method for Normal with exact_only. Tried: exact: Normal has no exact expectation
selected,methods,monte_carlo,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


## 5. The hierarchical model three ways

The non-centered eight-schools model has `mu ~ N(0, 5)`, `tau ~ HalfCauchy(0, 5)`, `theta_tilde_j ~ N(0, 1)`, and `y_j ~ N(mu + tau theta_tilde_j, sigma_j)`.
The three representations below are conditioned on the observed effects, and each posterior is brought to the parameter record `{population: {mu, tau}, groups: {theta_tilde}}`, by a rename where the program names its parameters flat.

### Stan

`StanModel` is a `ConditionalDistribution` from the program's data block to the unnormalized posterior over its parameters.
The adapter reads both declarations from the program before any data are bound.
Each data entry declares its dtype, each parameter declares its support, and `J` stays a symbolic dimension.

In [34]:
STAN_PROGRAM = """
data {
  int<lower=1> J;
  vector[J] y;
  vector<lower=0>[J] sigma;
}
parameters {
  real mu;
  real<lower=0> tau;
  vector[J] theta_tilde;
}
model {
  mu ~ normal(0, 5);
  tau ~ cauchy(0, 5);
  theta_tilde ~ normal(0, 1);
  y ~ normal(mu + tau * theta_tilde, sigma);
}
"""
stan_file = Path(tempfile.mkdtemp()) / "eight_schools.stan"
stan_file.write_text(STAN_PROGRAM)
stan_model = StanModel("eight_schools", str(stan_file))
stan_data = {"J": J, "y": np.asarray(y_obs), "sigma": np.asarray(sigma)}
pd.concat({"given": spec_table(stan_model.given_spec),
           "parameter": spec_table(stan_model.event_spec.spec)})

shape    dtype   support
given     J              ()    int32      None
          y            (J,)  float32      None
          sigma        (J,)  float32      None
parameter mu             ()  float32      real
          tau            ()  float32  positive
          theta_tilde  (J,)  float32      real

Binding the data curries the kernel to its unnormalized posterior, and `condition_on` normalizes it with nutpie, which is the highest-ranked method that applies.

In [35]:
condition_on.check(stan_model, stan_data)

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,"the exact stage's result is unnormalized, and no exact method normalizes it; method=""unnormalized"" returns that result: No method registered for _StanPosterior with exact_only"
,slice,exact methods,exact,infeasible,route 'slice' declined: the conditioned object is not a joint law
,exact_conditioning,,exact,infeasible,StanModel does not claim SupportsExactConditioning
,bayes,exact methods,exact,infeasible,route 'bayes' declined: the given names no produced field
selected,curry,nutpie_nuts,approximate,feasible,


The posterior is empirical over the draws, and its repr names the method that ran.
nutpie reads `progress_bar` among its method options.

In [36]:
NUTS = {"num_warmup": 500, "num_results": 500, "num_chains": 2}
NUTPIE = {**NUTS, "progress_bar": False}
with workflow_run(seed=1):
    stan_fit = condition_on.with_options(method_options=NUTPIE)(stan_model, stan_data)
stan_fit

ApproximateDistribution('eight_schools', method='nutpie_nuts', num_chains=2, num_draws=500)

Renaming the posterior nests its fields.

In [37]:
stan_posterior = stan_fit.with_path_names(NEST)
stan_posterior

ApproximateDistribution(
    'eight_schools',
    method='nutpie_nuts',
    num_chains=2,
    num_draws=500,
    event_spec=OutputSpec(
        NumericRecordSpec(
            population=NumericRecordSpec(
                mu=NumericArraySpec(shape=(), dtype=float32, support=real),
                tau=NumericArraySpec(shape=(), dtype=float32, support=positive),
            ),
            groups=NumericRecordSpec(
                theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
            ),
        ),
    ),
)

### PyMC

`PyMCModel` is the joint law that a model-building function defines over its free variables.
An argument that defaults to `None` and feeds an observed variable is an event field, so this model is a `Distribution` over `{mu, tau, theta_tilde, y}`.

In [38]:
def eight_schools_pymc(y=None):
    import pymc as pm

    with pm.Model() as model:
        mu = pm.Normal("mu", 0.0, 5.0)
        tau = pm.HalfCauchy("tau", 5.0)
        theta_tilde = pm.Normal("theta_tilde", 0.0, 1.0, shape=J)
        pm.Normal("y", mu + tau * theta_tilde, np.asarray(sigma), observed=y, shape=J)
    return model


pymc_model = PyMCModel("eight_schools", eight_schools_pymc)
print("pymc_model:", repr(pymc_model))
spec_table(pymc_model.event_spec.spec)

pymc_model: PyMCModel('eight_schools', model_fn=eight_schools_pymc, variables=('mu', 'tau', 'theta_tilde', 'y'))


,shape,dtype,support
mu,(),float32,real
tau,(),float32,positive
theta_tilde,"(8,)",float32,real
y,"(8,)",float32,real


Conditioning on `y` is Bayes' rule, normalized with nutpie.

In [39]:
condition_on.check(pymc_model, {"y": np.asarray(y_obs)})

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,route 'curry' declined: the conditioned object is not a kernel
,slice,exact methods,exact,infeasible,route 'slice' declined: the conditioned object is not a joint law
,exact_conditioning,,exact,infeasible,PyMCModel does not claim SupportsExactConditioning
,bayes,exact methods,exact,infeasible,"the exact stage's result is unnormalized, and no exact method normalizes it; method=""unnormalized"" returns that result: No method registered for _UnnormalizedConditional with exact_only"
,curry,approximate methods,approximate,infeasible,route 'curry' declined: the conditioned object is not a kernel
,slice,approximate methods,approximate,infeasible,route 'slice' declined: the conditioned object is not a joint law
,approximate_conditioning,,approximate,infeasible,PyMCModel does not claim SupportsApproximateConditioning
selected,bayes,nutpie_nuts,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


The fit runs nutpie, and the rename nests the posterior's fields.

In [40]:
with workflow_run(seed=1):
    pymc_fit = condition_on.with_options(method_options=NUTPIE)(pymc_model, {"y": np.asarray(y_obs)})
pymc_posterior = pymc_fit.with_path_names(NEST)
show({"pymc_fit": repr(pymc_fit), "fields of pymc_posterior": list(pymc_posterior.event_spec.spec)})

pymc_fit                  ApproximateDistribution('eight_schools', method='nutpie_nuts', num_chains=2, num_draws=500)
fields of pymc_posterior  ['population/mu', 'population/tau', 'groups/theta_tilde']


Renaming a program family before conditioning nests its parameters and keeps its route.

In [41]:
programs = {"Stan": (stan_model, stan_data), "PyMC": (pymc_model, {"y": np.asarray(y_obs)})}
pd.DataFrame(
    {"route of the model": [condition_on.check(model, given).selected.method_name
                            for model, given in programs.values()],
     "route of the renamed model": [condition_on.check(model.with_path_names(NEST), given).selected.method_name
                                    for model, given in programs.values()]},
    index=list(programs),
)

,route of the model,route of the renamed model
Stan,curry/nutpie_nuts,curry/nutpie_nuts
PyMC,bayes/nutpie_nuts,bayes/nutpie_nuts


### A native factored joint

`conditional_distribution` builds a native likelihood from a function of its given values that returns a law.
Its given slots are the prior's two groups, each of which arrives as a `NumericRecord`, and the kernel samples and scores through the law the function returns.

In [42]:
@conditional_distribution("y", given_spec=prior.event_spec.components)
def likelihood(population, groups):
    return Normal("y", population["mu"] + population["tau"] * groups["theta_tilde"], sigma)


show({"likelihood": repr(likelihood), "given slots": list(likelihood.given_spec),
      "event": likelihood.event_spec.spec})

likelihood   ConditionalDistribution('y', given=('population', 'groups'))
given slots  ['population', 'groups']
event        NumericArraySpec(shape=(8,), dtype=float32, support=real)


Composition is conditional-first: the likelihood conditions on what the prior produces, so `likelihood * prior` is a `FactoredDistribution` over `{y, population, groups}`, labeled by joining the operands' labels with `·`.

In [43]:
schools = likelihood * prior
schools

FactoredDistribution(
    'y·mu·tau·theta_tilde',
    factors=(
        ConditionalDistribution('y', given=('population', 'groups')),
        FactoredDistribution(
            'mu·tau',
            factors=(Normal('mu', loc=0.0, scale=5.0), HalfCauchy('tau', loc=0.0, scale=5.0)),
            event_spec=OutputSpec(
                population=NumericRecordSpec(
                    mu=NumericArraySpec(shape=(), dtype=float32, support=real),
                    tau=NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0)),
                ),
            ),
        ),
        FactoredDistribution(
            'theta_tilde',
            factors=(
                Normal('theta_tilde', loc=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], scale=1.0),
            ),
            event_spec=OutputSpec(
                groups=NumericRecordSpec(
                    theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
                ),
            ),
        ),
    ),
)

The joint samples ancestrally and scores a draw.

In [44]:
with workflow_run(seed=0):
    joint_draw = sample(schools)
show({"joint_draw": repr(joint_draw),
      "log_prob(schools, joint_draw)": float(log_prob(schools, joint_draw))})

joint_draw                     NumericRecord(
                                   'y·mu·tau·theta_tilde',
                                   fields=('y', 'population/mu', 'population/tau', 'groups/theta_tilde'),
                               )
log_prob(schools, joint_draw)  -45.360511779785156


Conditioning on `y` resolves to Bayes' rule, and the inference-method registry selects BlackJAX's NUTS.

In [45]:
condition_on.check(schools, {"y": y_obs})

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,route 'curry' declined: the conditioned object is not a kernel
,slice,exact methods,exact,infeasible,"route 'slice' declined: the factor producing ['y'] conditions on ['groups', 'population'], which the given leaves free, so the conditional is not a product of the factors"
,exact_conditioning,,exact,infeasible,FactoredDistribution does not claim SupportsExactConditioning
,bayes,exact methods,exact,infeasible,"the exact stage's result is unnormalized, and no exact method normalizes it; method=""unnormalized"" returns that result: No method registered for _UnnormalizedConditional with exact_only"
,curry,approximate methods,approximate,infeasible,route 'curry' declined: the conditioned object is not a kernel
,slice,approximate methods,approximate,infeasible,"route 'slice' declined: the factor producing ['y'] conditions on ['groups', 'population'], which the given leaves free, so the conditional is not a product of the factors"
,approximate_conditioning,,approximate,infeasible,FactoredDistribution does not claim SupportsApproximateConditioning
selected,bayes,blackjax_nuts,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


TFP's NUTS runs when `method=` names it, and each posterior's repr names its method.

In [46]:
with workflow_run(seed=1):
    native_posterior = condition_on.with_options(method_options=NUTS)(schools, {"y": y_obs})
    tfp_posterior = condition_on.with_options(method="tfp_nuts", method_options=NUTS)(schools, {"y": y_obs})
show({"native_posterior": repr(native_posterior), "tfp_posterior": repr(tfp_posterior)})

native_posterior  ApproximateDistribution('y·mu·tau·theta_tilde', method='blackjax_nuts', num_chains=2, num_draws=500)
tfp_posterior     ApproximateDistribution('y·mu·tau·theta_tilde', method='tfp_nuts', num_chains=2, num_draws=500)


### Comparison

The school effects `theta_j = mu + tau theta_tilde_j` are a function of three fields of the posterior.
Passing the posterior's field views lifts the function, and the views co-sample, so each evaluation reads one posterior draw.
With `n_broadcast_samples` at the atom count, the lift enumerates each empirical posterior exactly, a renamed one included.

In [47]:
@function
def school_effects(mu, tau, theta_tilde):
    return mu + tau * theta_tilde


posteriors = {
    "Stan, nutpie": stan_posterior,
    "PyMC, nutpie": pymc_posterior,
    "native, BlackJAX NUTS": native_posterior,
    "native, TFP NUTS": tfp_posterior,
}
summary_rows = {}
for label, post in posteriors.items():
    with workflow_run(seed=4):
        thetas = school_effects.with_options(n_broadcast_samples=1000)(
            post["population/mu"], post["population/tau"], post["groups/theta_tilde"]
        )
    means, variances = mean(post), variance(post)
    summaries = [
        ("mu", means["population/mu"], variances["population/mu"]),
        ("tau", means["population/tau"], variances["population/tau"]),
        *[(f"theta[{j}]", mean(thetas).value[j], variance(thetas).value[j]) for j in range(3)],
    ]
    summary_rows[label] = {(name, stat): value for name, m, v in summaries
                           for stat, value in (("mean", float(m)), ("sd", float(v) ** 0.5))}
    summary_rows[label][("lift", "route")] = thetas.provenance.metadata["route"]
pd.DataFrame.from_dict(summary_rows, orient="index").round(2)

mu         tau       theta[0]       theta[1]        \
                       mean    sd  mean    sd     mean    sd     mean    sd   
Stan, nutpie           4.33  3.39  3.60  3.11     6.09  5.63     4.89  4.80   
PyMC, nutpie           4.42  3.32  3.64  3.34     6.41  5.64     4.94  5.02   
native, BlackJAX NUTS  4.22  2.99  2.87  2.38     5.49  4.85     4.49  4.42   
native, TFP NUTS       4.32  3.28  3.07  3.11     5.81  5.35     4.76  4.40   

                      theta[2]                         lift  
                          mean    sd                  route  
Stan, nutpie              3.85  5.23  empirical_enumeration  
PyMC, nutpie              4.04  5.45  empirical_enumeration  
native, BlackJAX NUTS     3.95  4.31  empirical_enumeration  
native, TFP NUTS          3.77  5.01  empirical_enumeration

### Declarations, provenance, and controls

Each posterior declares its target's event, with the Stan dimension `J` bound.

In [48]:
pd.concat({"Stan target": spec_table(stan_model.event_spec.spec),
           "Stan posterior": spec_table(stan_fit.event_spec.spec)}, axis=1)

Stan target                    Stan posterior                   
                  shape    dtype   support          shape    dtype   support
mu                   ()  float32      real             ()  float32      real
tau                  ()  float32  positive             ()  float32  positive
theta_tilde        (J,)  float32      real           (8,)  float32      real

Each posterior's provenance records the route, the method, and whether the step was exact.

In [49]:
runs = {"Stan": stan_fit, "PyMC": pymc_fit, "native, BlackJAX": native_posterior, "native, TFP": tfp_posterior}
pd.DataFrame([run.provenance.metadata for run in runs.values()], index=list(runs))

,func,route,exact,method
Stan,condition_on,curry,False,nutpie_nuts
PyMC,condition_on,bayes,False,nutpie_nuts
"native, BlackJAX",condition_on,bayes,False,blackjax_nuts
"native, TFP",condition_on,bayes,False,tfp_nuts


`with_options` sets an operation's controls, and `options` reports the effective value of each.

In [50]:
strict = condition_on.with_options(exact_only=True)
pd.DataFrame({"condition_on": {name: str(value) for name, value in condition_on.options.items()},
              "with exact_only=True": {name: str(value) for name, value in strict.options.items()}})

,condition_on,with exact_only=True
workflow_kind,WorkflowKind.DEFAULT,WorkflowKind.DEFAULT
n_broadcast_samples,128,128
dispatch,auto,auto
max_workers,None,None
include_inputs,False,False
method,None,None
exact_only,False,True
conversions,{},{}
method_options,{},{}
raw,False,False


`exact_only=True` excludes every approximate route.
No exact method normalizes the unnormalized posterior, so the call raises `ResolutionError`, and the report names `method="unnormalized"`.

In [51]:
try:
    strict(stan_model, stan_data)
except ResolutionError as error:
    print("the call raises:", type(error).__name__)
strict.check(stan_model, stan_data)

the call raises: ResolutionError


,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,"the exact stage's result is unnormalized, and no exact method normalizes it; method=""unnormalized"" returns that result: No method registered for _StanPosterior with exact_only"
,slice,exact methods,exact,infeasible,route 'slice' declined: the conditioned object is not a joint law
,exact_conditioning,,exact,infeasible,StanModel does not claim SupportsExactConditioning
,bayes,exact methods,exact,infeasible,route 'bayes' declined: the given names no produced field


`method="unnormalized"` returns the exact stage alone: the unnormalized posterior, which claims only `SupportsUnnormalizedLogProb`.
The two densities differ by the constants that Stan's sampling statements drop.

In [52]:
stan_target = condition_on.with_options(method="unnormalized")(stan_model, stan_data)
native_target = condition_on.with_options(method="unnormalized")(schools, {"y": y_obs})
targets = {
    "Stan": (stan_target, {"mu": 4.0, "tau": 3.0, "theta_tilde": np.zeros(J)}),
    "native": (native_target, {"population": {"mu": 4.0, "tau": 3.0}, "groups": {"theta_tilde": jnp.zeros(J)}}),
}
claims = (SupportsSampling, SupportsLogProb, SupportsUnnormalizedLogProb)
pd.DataFrame(
    {**{p.__name__: [isinstance(target, p) for target, _ in targets.values()] for p in claims},
     "unnormalized_log_prob at mu=4, tau=3": [float(unnormalized_log_prob(target, point))
                                             for target, point in targets.values()]},
    index=list(targets),
)

,SupportsSampling,SupportsLogProb,SupportsUnnormalizedLogProb,"unnormalized_log_prob at mu=4, tau=3"
Stan,False,False,True,-3.390654
native,False,False,True,-42.652267


Conditioning on the upstream field `population` is the exact slice `p(y | population, groups) p(groups)`, since the regrouped prior stays factored and the likelihood curries over `groups`.

In [53]:
population_value = {"mu": 4.0, "tau": 3.0}
condition_on.check(schools, {"population": population_value})

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,route 'curry' declined: the conditioned object is not a kernel
selected,slice,exact methods,exact,feasible,
,,,,deferred,the result's type is completed from the returned value


The slice is a factored joint over `{y, groups}`, whose likelihood has the one given slot `groups` left.

In [54]:
at_population = condition_on(schools, {"population": population_value})
at_population

FactoredDistribution(
    'y·mu·tau·theta_tilde',
    factors=(
        ConditionalDistribution('y', given=('groups',)),
        FactoredDistribution(
            'theta_tilde',
            factors=(
                Normal('theta_tilde', loc=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], scale=1.0),
            ),
            event_spec=OutputSpec(
                groups=NumericRecordSpec(
                    theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
                ),
            ),
        ),
    ),
)

## 6. Field views and marginals

`d[path]` is a field view: a law over the field at `path` that refers to its parent.
A view of a leaf works on the joint and on the posterior.
**Transitional:** a view is labeled by its path, and it keeps its parent's label once the label fix is merged (Z5).

In [55]:
tau_view = schools["population/tau"]
print('tau_view, "| label:", tau_view.name, "| event:", tau_view.event_spe...:', repr(tau_view), "| label:", tau_view.name, "| event:", tau_view.event_spec.spec)
print('native_posterior["groups/theta_tilde"]:', repr(native_posterior["groups/theta_tilde"]))
print('mean(native_posterior["groups/theta_tilde"]):', repr(mean(native_posterior["groups/theta_tilde"])))

tau_view, "| label:", tau_view.name, "| event:", tau_view.event_spe...: FieldView('y·mu·tau·theta_tilde', path='population/tau') | label: y·mu·tau·theta_tilde | event: NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0))
native_posterior["groups/theta_tilde"]: FieldView('y·mu·tau·theta_tilde', path='groups/theta_tilde')
mean(native_posterior["groups/theta_tilde"]): NumericArray('y·mu·tau·theta_tilde', shape=(8,), dtype=float32, support=real)


**Known bug:** `d[path]` at a regrouped node raises, so `population`, which the rename gathered from two fields, has no view.

In [56]:
try:
    schools["population"]
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

A view derives its capabilities from its parent's.
The view of `y` claims `SupportsLogProb`, but its guard declines, since the marginal density of `y` integrates out the parameters that `y` conditions on.

In [57]:
y_view = schools["y"]
print('isinstance(y_view, SupportsLogProb):', isinstance(y_view, SupportsLogProb))
try:
    log_prob(y_view, y_obs)
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")
print('log_prob(tau_view, 1.0):', repr(log_prob(tau_view, 1.0)))

isinstance(y_view, SupportsLogProb): True
ResolutionError: log_prob: no route applies. Tried: exact: the marginal at ['y'] integrates out the fields ['population', 'groups'], which the requested fields condition on
log_prob(tau_view, 1.0): NumericArray('y·mu·tau·theta_tilde', shape=(), dtype=float32)


**Finding:** the message names the factor label `mu·tau·theta_tilde` rather than the fields `population` and `groups`.
**Open question:** a public capability for the density of a subset of observations, such as one school's `y_j`.

Sibling views co-sample when lifted together, since the lift groups arguments by their root and draws the parent once per repetition, while detached marginals draw independently.
The standardized residual `(y - mu - tau theta_tilde) / sigma` is standard normal under the joint, so its spread shows which happened.
Two views of one field, built separately, also co-sample.

In [58]:
@function
def residual(y, mu, tau, theta_tilde):
    return (y - mu - tau * theta_tilde) / sigma


@function
def difference(a, b):
    return a - b


fields = ["population/mu", "population/tau", "groups/theta_tilde"]
with workflow_run(seed=0):
    from_views = residual(schools["y"], *(schools[path] for path in fields))
    from_marginals = residual(schools["y"], *(marginal(schools, path) for path in fields))
    same_field = difference(schools["population/mu"], schools["population/mu"])
print("residual sd, sibling views:     ", float(jnp.std(from_views.atoms.values)))
print("residual sd, detached marginals:", float(jnp.std(from_marginals.atoms.values)))
print("one field viewed twice: max |a - b| =", float(jnp.abs(same_field.atoms.values).max()))

residual sd, sibling views:      1.0353190898895264
residual sd, detached marginals: 7.557694911956787
one field viewed twice: max |a - b| = 0.0


`marginal` returns the detached marginal.
A root field's marginal is its exact factor, while a field that the likelihood produces has no exact route.
**Transitional:** the marginal is labeled `marginal`, and it keeps its parent's label after the fix (Z5).

In [59]:
mu_marginal = marginal(schools, "population/mu")
print(repr(mu_marginal), mu_marginal.event_spec.spec, mu_marginal.provenance.metadata)
show_report(marginal.check(schools, "y"))
try:
    marginal(schools, "y")
except NotImplementedError as error:
    print(f"{type(error).__name__}: {error}")

Normal(
    'y·mu·tau·theta_tilde',
    loc=0.0,
    scale=5.0,
    event_spec=OutputSpec(mu=NumericArraySpec(shape=(), dtype=float32, support=real)),
) NumericArraySpec(shape=(), dtype=float32, support=real) {'func': 'marginal', 'route': 'exact', 'exact': True}


,route,method,exact,feasible,reason
,exact,,exact,infeasible,"the marginal at ['y'] integrates out the fields ['population', 'groups'], which the requested fields condition on"
,monte_carlo,,approximate,infeasible,the empirical marginal of projected draws is not implemented


ResolutionError: marginal: no route applies. Tried: exact: the marginal at ['y'] integrates out the fields ['population', 'groups'], which the requested fields condition on; monte_carlo: the empirical marginal of projected draws is not implemented

**Finding:** `check` selects the `monte_carlo` route for `y`, and its execution raises `NotImplementedError`; `tests/correctness/test_models.py:501` marks the case pending.

On the empirical posterior every marginal is exact, a selection included, and the selection's fields follow the order of its paths.

In [60]:
pair = marginal(native_posterior, ("groups/theta_tilde", "population/mu"))
print('pair, pair.event_spec.spec:', repr(pair), pair.event_spec.spec)
print('native_posterior[("groups/theta_tilde", "population/mu")]:', repr(native_posterior[("groups/theta_tilde", "population/mu")]))

pair, pair.event_spec.spec: EmpiricalDistribution(
    'y·mu·tau·theta_tilde',
    atoms=NumericRecordBatch(
        "posterior[('groups/theta_tilde', 'population/mu')]",
        levels={'chain': 2, 'draw': 500},
        fields=('theta_tilde', 'mu'),
    ),
) NumericRecordSpec(
    theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
    mu=NumericArraySpec(shape=(), dtype=float32, support=real),
)
native_posterior[("groups/theta_tilde", "population/mu")]: FieldView('y·mu·tau·theta_tilde', path=('groups/theta_tilde', 'population/mu'))


**Known bug:** on a factored joint, a selection's marginal orders its fields by factor, while the view keeps the requested order.

In [61]:
ab = Normal("a", 0.0, 1.0) * Normal("b", 5.0, 1.0)
print("view:    ", ab[("b", "a")].event_spec.spec)
print("marginal:", marginal(ab, ("b", "a")).event_spec.spec)

view:     NumericRecordSpec(
    b=NumericArraySpec(shape=(), dtype=float32, support=real),
    a=NumericArraySpec(shape=(), dtype=float32, support=real),
)
marginal: NumericRecordSpec(
    b=NumericArraySpec(shape=(), dtype=float32, support=real),
    a=NumericArraySpec(shape=(), dtype=float32, support=real),
)


**Finding:** a view and a marginal of one field answer a moment differently.
The joint is dependent, so the view's mean takes the Monte Carlo route and returns a number for `tau`, whose mean does not exist, while the exact marginal raises `MathematicalDomainError`.

In [62]:
with workflow_run(seed=0):
    print(repr(mean(tau_view)), "via", mean.check(tau_view).selected.method_name)
try:
    mean(marginal(schools, "population/tau"))
except MathematicalDomainError as error:
    print(f"{type(error).__name__}: {error}")

NumericArray('y·mu·tau·theta_tilde', shape=(), dtype=float32, support=greater_than(0.0)) via closed_form


`factor` returns the building block that produces a component: the likelihood for `y`, and the renamed prior for `groups`.

In [63]:
print('factor(schools, component_name="y"):', repr(factor(schools, component_name="y")))
print('factor(schools, component_name="groups"):', repr(factor(schools, component_name="groups")))

factor(schools, component_name="y"): ConditionalDistribution('y', given=('population', 'groups'))
factor(schools, component_name="groups"): FactoredDistribution(
    'theta_tilde',
    factors=(Normal('theta_tilde', loc=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], scale=1.0),),
    event_spec=OutputSpec(
        groups=NumericRecordSpec(
            theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
        ),
    ),
)


## 7. Functions and lifting

`@function` wraps a plain callable, and a call on values evaluates it and records provenance.
The annotations decide lifting: a parameter that is unannotated or annotated with a value type lifts a law, while a `Distribution` or `Any` parameter receives the law whole.

In [64]:
@function
def effect_of(mu: float, tau: float, z: float) -> float:
    return mu + tau * z


value = effect_of(4.0, 3.0, 0.5)
print(repr(value), value.provenance)

NumericArray('effect_of', shape=(), dtype=float32) Provenance('workflow.effect_of', parents=[effect_of])


A law at a value parameter is the sampling lift, whose result is the pushforward: an empirical law over the outputs.
`check` reports the route and the lifted parameters.

In [65]:
report = effect_of.check(Normal("mu", 0.0, 5.0), 3.0, 0.5)
show_report(report)
print("lifted:", report.lifted)
with workflow_run(seed=0):
    lifted = effect_of(Normal("mu", 0.0, 5.0), 3.0, 0.5)
print('lifted, lifted.event_spec.spec:', repr(lifted), lifted.event_spec.spec)
print('lifted.provenance.metadata:', lifted.provenance.metadata)

,route,method,exact,feasible,reason
selected,sampling_lift,,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


lifted: ('mu',)


lifted, lifted.event_spec.spec: EmpiricalDistribution(
    'effect_of',
    atoms=NumericArrayBatch(
        'effect_of',
        levels={'draw': 128},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
) NumericArraySpec(shape=(), dtype=float32)
lifted.provenance.metadata: {'dispatch': 'jax', 'orchestrate': 'off', 'n_samples': 128, 'func': 'effect_of', 'broadcast_args': ['mu'], 'route': 'sampling_lift', 'exact': False}


A batch at a value parameter is swept: the function maps over the elements and keeps the batch's level.

In [66]:
taus = NumericArrayBatch("tau", jnp.array([0.0, 1.0, 5.0, 10.0]), "tau", element_spec=NumericArraySpec((), jnp.float32))
swept = effect_of(4.0, taus, 0.5)
print(repr(swept), swept.values)

NumericArrayBatch(
    'effect_of',
    levels={'tau': 4},
    element_spec=NumericArraySpec(shape=(), dtype=float32),
) [4.  4.5 6.5 9. ]


A law and a batch together give a nested sweep of broadcasts: one pushforward per element, collected in a `DistributionBatch` on the batch's level.

In [67]:
with workflow_run(seed=0):
    nested = effect_of.with_options(n_broadcast_samples=64)(Normal("mu", 0.0, 5.0), taus, 0.5)
print('nested:', repr(nested))
print('nested[3]:', repr(nested[3]))
print('mean(nested), mean(nested).values:', repr(mean(nested)), mean(nested).values)

nested: DistributionBatch(
    'effect_of',
    levels={'tau': 4},
    element_spec=DistributionSpec(
        event_spec=OutputSpec(effect_of=NumericArraySpec(shape=(), dtype=float32)),
    ),
)


nested[3]: EmpiricalDistribution(
    'effect_of[tau=3]',
    atoms=NumericArrayBatch(
        'effect_of',
        levels={'draw': 64},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
    event_spec=OutputSpec(effect_of=NumericArraySpec(shape=(), dtype=float32)),
)
mean(nested), mean(nested).values: NumericArrayBatch(
    'effect_of',
    levels={'tau': 4},
    element_spec=NumericArraySpec(shape=(), dtype=float32),
) [ 0.068247   -0.01316434  2.5583198   4.606876  ]


`include_inputs=True` returns the joint empirical law over the lifted inputs and the output, so each atom pairs an input draw with its output.

In [68]:
with workflow_run(seed=0):
    with_inputs = effect_of.with_options(include_inputs=True, n_broadcast_samples=64)(
        Normal("mu", 0.0, 5.0), HalfCauchy("tau", 0.0, 5.0), 0.5
    )
print(repr(with_inputs), with_inputs.event_spec.spec)

EmpiricalDistribution(
    'effect_of',
    atoms=NumericRecordBatch('effect_of', levels={'draw': 64}, fields=('mu', 'tau', 'effect_of')),
) NumericRecordSpec(
    mu=NumericArraySpec(shape=(), dtype=float32, support=real),
    tau=NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0)),
    effect_of=NumericArraySpec(shape=(), dtype=float32),
)


An empirical law whose atoms number at most `n_broadcast_samples` is enumerated exactly: each atom is evaluated once and keeps its weight.
Two empirical laws enumerate together, in either argument order.

In [69]:
tau_atoms = EmpiricalDistribution("tau", jnp.array([0.0, 1.0, 5.0]), weights=jnp.array([0.2, 0.3, 0.5]))
enumerated = effect_of(4.0, tau_atoms, 0.5)
print('enumerated, enumerated.atoms.values, enumerated.weights:', repr(enumerated), enumerated.atoms.values, enumerated.weights)
print('enumerated.provenance.metadata["route"], enumerated.provenance.meta...:', enumerated.provenance.metadata["route"], enumerated.provenance.metadata["exact"])
z_atoms = EmpiricalDistribution("z", jnp.array([-1.0, 1.0]))
for args in [(4.0, tau_atoms, z_atoms), (z_atoms, tau_atoms, 0.5)]:
    result = effect_of(*args)
    print(result.num_atoms, result.provenance.metadata["route"], result.provenance.metadata["exact"])

enumerated, enumerated.atoms.values, enumerated.weights: EmpiricalDistribution(
    'effect_of',
    atoms=NumericArrayBatch(
        'effect_of',
        levels={'draw': 3},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
    weights=[0.19999999, 0.29999998, 0.5],
) [4.  4.5 6.5] [0.19999999 0.29999998 0.5       ]
enumerated.provenance.metadata["route"], enumerated.provenance.meta...: empirical_enumeration True


6 empirical_enumeration True
6 empirical_enumeration True


`n_broadcast_samples` sets the number of draws of the sampling lift, and its default is 128.
With an empirical law and a continuous one, the lift enumerates the atoms and samples the continuous law within each.

In [70]:
with workflow_run(seed=0):
    print(repr(effect_of.with_options(n_broadcast_samples=1000)(Normal("mu", 0.0, 5.0), 3.0, 0.5)))
    mixed = effect_of(Normal("mu", 0.0, 5.0), tau_atoms, 0.5)
print(repr(mixed), mixed.provenance.metadata["route"], mixed.weights[:2], mixed.weights[-2:])

EmpiricalDistribution(
    'effect_of',
    atoms=NumericArrayBatch(
        'effect_of',
        levels={'draw': 1000},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
)


EmpiricalDistribution(
    'effect_of',
    atoms=NumericArrayBatch(
        'effect_of',
        levels={'draw': 126},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
    weights=array(shape=(126,), dtype=float32),
) sampling_lift [0.0047619 0.0047619] [0.01190476 0.01190476]


**Finding:** the mixed lift draws 42 values per atom, so 128 samples become 126 atoms weighted by the atom weights; the design does not describe this scheme, and the result reports the route `sampling_lift`.

An `Any` parameter receives the law whole, as a `Distribution` parameter does.

In [71]:
@function
def density_at(law: Any, x: float) -> float:
    return log_prob.with_options(raw=True)(law, x)


print(repr(density_at(Normal("mu", 0.0, 5.0), 1.0)))

NumericArray('density_at', shape=(), dtype=float32)


A transformed law co-samples with its base, since both share one root.
`BijectorTransformedDistribution` pushes a base law through a bijector, and here the bijector is the canonical map onto the positive half-line.

In [72]:
log_tau = Normal("log_tau", 1.0, 0.5)
tau_lognormal = BijectorTransformedDistribution("tau", log_tau, bijector_for(positive))
print('tau_lognormal, tau_lognormal.event_spec.spec:', repr(tau_lognormal), tau_lognormal.event_spec.spec)


@function
def mismatch(log_t, t):
    return t - jnp.exp(log_t)


with workflow_run(seed=0):
    paired = mismatch(log_tau, tau_lognormal)
    unpaired = mismatch(Normal("other", 1.0, 0.5), tau_lognormal)
print("with its base:   max |t - exp(log_t)| =", float(jnp.abs(paired.atoms.values).max()))
print("with another law: max |t - exp(log_t)| =", float(jnp.abs(unpaired.atoms.values).max()))

tau_lognormal, tau_lognormal.event_spec.spec: BijectorTransformedDistribution(
    'tau',
    base=Normal('log_tau', loc=1.0, scale=0.5),
    bijector=Function('exp', parameters=('x',)),
) NumericArraySpec(shape=(), dtype=float32, support=positive)
with its base:   max |t - exp(log_t)| = 0.0
with another law: max |t - exp(log_t)| = 11.39038372039795


`evaluate(f, d, fixed_args=...)` is the operation form of the call, and it takes the same route.

In [73]:
with workflow_run(seed=0):
    evaluated = evaluate(effect_of, Normal("mu", 0.0, 5.0), fixed_args={"tau": 3.0, "z": 0.5})
print('evaluated, evaluated.provenance.metadata:', repr(evaluated), evaluated.provenance.metadata)
show_report(evaluate.check(effect_of, Normal("mu", 0.0, 5.0), fixed_args={"tau": 3.0, "z": 0.5}))
try:
    evaluate.with_options(method="sampling_lift")(effect_of, Normal("mu", 0.0, 5.0), fixed_args={"tau": 3.0, "z": 0.5})
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")
print('probpipe.evaluation_rule_registry.list_methods():', probpipe.evaluation_rule_registry.list_methods())

evaluated, evaluated.provenance.metadata: EmpiricalDistribution(
    'effect_of',
    atoms=NumericArrayBatch(
        'effect_of',
        levels={'draw': 128},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
) {'func': 'evaluate', 'route': 'evaluation_rules', 'exact': False, 'method': 'sampling_lift'}


,route,method,exact,feasible,reason
,evaluation_rules,exact methods,exact,infeasible,"No feasible method for (Function, Normal) with exact_only. Tried: empirical_enumeration: Normal 'mu' is not an empirical law"
selected,evaluation_rules,sampling_lift,approximate,feasible,
,,,,deferred,the result's type is completed from the returned value


probpipe.evaluation_rule_registry.list_methods(): ['empirical_enumeration', 'elementwise_sweep', 'sampling_lift']


**Known bug:** `evaluate.with_options(method=...)` cannot name a rule.
**Finding:** `evaluate`'s `check` reports the one route `evaluation_rules` and not the rule it selects, and the result is labeled `evaluate` where the direct call's is labeled `effect_of`.
The registry holds three rules; the closed-form, change-of-variables, and linear-pushforward rules of V.7 are not registered yet, and `LinearPushforwardDistribution.__init__` raises `NotImplementedError`.

## 8. Batching

A `DistributionBatch` holds separate laws that share one event declaration.
The component defaults to the label, so the elements share a label as well.

In [74]:
school_laws = DistributionBatch("schools", [Normal("effect", y, s) for y, s in zip(y_obs, sigma)], "school")
print('school_laws, school_laws.event_spec.spec:', repr(school_laws), school_laws.event_spec.spec)
print('school_laws[0]:', repr(school_laws[0]))
try:
    DistributionBatch("pair", [Normal("a", 0.0, 1.0), Normal("b", 0.0, 1.0)], "law")
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

school_laws, school_laws.event_spec.spec: DistributionBatch(
    'schools',
    levels={'school': 8},
    element_spec=DistributionSpec(
        event_spec=OutputSpec(effect=NumericArraySpec(shape=(), dtype=float32, support=real)),
    ),
) NumericArraySpec(shape=(), dtype=float32, support=real)
school_laws[0]: Normal(
    'schools[school=0]',
    loc=28.0,
    scale=15.0,
    event_spec=OutputSpec(effect=NumericArraySpec(shape=(), dtype=float32, support=real)),
)
TypeError: every element of a DistributionBatch must be a Distribution whose event declaration matches the batch's, and the law at 1 departs from it: the batch declares the components ['a'] and the law ['b']


**Finding:** the refusal does not say that the components `a` and `b` differ.

Operations lift over the batch: `sample` nests its draws inside the laws' level, and moments and densities keep the level.

In [75]:
with workflow_run(seed=0):
    batch_draws = sample(school_laws, sample_shape=(5,))
print('batch_draws, batch_draws.axis_groups:', repr(batch_draws), batch_draws.axis_groups)
print('mean(school_laws), mean(school_laws).values:', repr(mean(school_laws)), mean(school_laws).values)
print('log_prob(school_laws, 0.0):', repr(log_prob(school_laws, 0.0)))
intervals = quantile(school_laws, jnp.array([0.05, 0.95]))
print('intervals:', repr(intervals))
print('intervals.values[:2]:', intervals.values[:2])

batch_draws, batch_draws.axis_groups: NumericArrayBatch(
    'schools',
    levels={'school': 8, 'sample': 5},
    element_spec=NumericArraySpec(shape=(), dtype=float32, support=real),
) ((8,), (5,))
mean(school_laws), mean(school_laws).values: NumericArrayBatch(
    'schools',
    levels={'school': 8},
    element_spec=NumericArraySpec(shape=(), dtype=float32, support=real),
) [28.  8. -3.  7. -1.  1. 18. 12.]
log_prob(school_laws, 0.0): NumericArrayBatch('schools', levels={'school': 8}, element_spec=NumericArraySpec(shape=()))


intervals: NumericArrayBatch(
    'schools',
    levels={'school': 8, 'quantile': 2},
    element_spec=NumericArraySpec(shape=(), dtype=float32),
)
intervals.values[:2]: [[ 3.3271961 52.672806 ]
 [-8.448536  24.448536 ]]


**Known bug:** two accesses of one batch element do not co-sample.
Each `school_laws[0]` is a new view, and the two views draw independently, while one object passed twice co-samples.

In [76]:
first = school_laws[0]
with workflow_run(seed=0):
    two_accesses = difference(school_laws[0], school_laws[0])
    one_object = difference(first, first)
print("two accesses: max |a - b| =", float(jnp.abs(two_accesses.atoms.values).max()))
print("one object:   max |a - b| =", float(jnp.abs(one_object.atoms.values).max()))

two accesses: max |a - b| = 0.0
one object:   max |a - b| = 0.0


A sweep over datasets gives a batch of posteriors.
`condition_on` does not sweep a batch inside its `given`, so a `@function` that conditions on one dataset is swept over a batch of three, with short chains.

In [77]:
datasets = NumericArrayBatch("y", jnp.stack([y_obs, 0.5 * y_obs, y_obs + 10.0]), "dataset", element_spec=NumericArraySpec((J,), jnp.float32))
SHORT = {"num_warmup": 200, "num_results": 200, "num_chains": 1}


@function
def fit(y):
    return condition_on.with_options(method_options=SHORT)(schools, {"y": y})


with workflow_run(seed=2):
    fits = fit(datasets)
print('fits, fits[1]:', repr(fits), repr(fits[1]))
print("posterior mean of mu by dataset:", mean(fits)["population/mu"])
givens = NumericRecordBatch("data", {"y": datasets.values}, "dataset", element_spec=RecordSpec(y=NumericArraySpec((J,), jnp.float32)))
try:
    condition_on(schools, givens)
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")

fits, fits[1]: DistributionBatch(
    'fit',
    levels={'dataset': 3},
    element_spec=DistributionSpec(
        event_spec=OutputSpec(
            NumericRecordSpec(
                population=NumericRecordSpec(
                    mu=NumericArraySpec(shape=(), dtype=float32, support=real),
                    tau=NumericArraySpec(shape=(), dtype=float32, support=greater_than(0.0)),
                ),
                groups=NumericRecordSpec(
                    theta_tilde=NumericArraySpec(shape=(8,), dtype=float32, support=real),
                ),
            ),
        ),
    ),
) ApproximateDistribution('fit[dataset=1]', method='blackjax_nuts', num_chains=1, num_draws=200)
posterior mean of mu by dataset: NumericArrayBatch(
    "fit['population/mu']",
    levels={'dataset': 3},
    element_spec=NumericArraySpec(shape=(), dtype=float32, support=real),
)


**Finding:** a `RecordBatch` of givens raises `ResolutionError`, while VI.11 lifts every operation over a batch.

`iterate` folds a step over inputs and returns the visited laws as a batch on a level named `iterate`.
Each step here updates a normal law on `mu` with one school's effect, treating `sigma_j` as known.

In [78]:
def update(law, school):
    effect, se = school
    m, v = float(mean(law).value), float(variance(law).value)
    precision = 1.0 / v + 1.0 / se**2
    return Normal("mu", (m / v + effect / se**2) / precision, precision**-0.5)


steps = iterate(update, Normal("mu", 0.0, 5.0), list(zip(y_obs.tolist(), sigma.tolist())))
print('steps:', repr(steps))
print('mean(steps).values:', mean(steps).values)

steps: DistributionBatch(
    'iterate',
    levels={'iterate': 9},
    element_spec=DistributionSpec(
        event_spec=OutputSpec(mu=NumericArraySpec(shape=(), dtype=float32, support=real)),
    ),
)
mean(steps).values: [0.        2.8       3.755102  3.302886  3.7615602 3.017081  2.8259656
 4.386674  4.620923 ]


## 9. Empirical laws and resampling

An `EmpiricalDistribution` is a weighted set of atoms.
A plain array's atoms lie on one level, named by `level`, which defaults to the component.
With precision weights, the mean is the pooled estimate of a common effect.

In [79]:
effects = EmpiricalDistribution("effect", y_obs, weights=1.0 / sigma**2, level="school")
print('effects, effects.atoms.level_names, effects.weights:', repr(effects), effects.atoms.level_names, effects.weights)
print('mean(effects), "pooled:", float(jnp.sum(y_obs / sigma**2) / jnp.sum...:', repr(mean(effects)), "pooled:", float(jnp.sum(y_obs / sigma**2) / jnp.sum(1.0 / sigma**2)))
print('[p.__name__ for p in (SupportsSampling, SupportsLogProb, SupportsMe...:', [p.__name__ for p in (SupportsSampling, SupportsLogProb, SupportsMean, SupportsMarginals) if isinstance(effects, p)])

effects, effects.atoms.level_names, effects.weights: EmpiricalDistribution(
    'effect',
    atoms=NumericArrayBatch(
        'effect',
        levels={'school': 8},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
    weights=[0.07369122, 0.16580525, 0.06476768, 0.13702913, 0.20469785, 0.13702913, 0.16580525, 0.05117446],
) ('school',) [0.07369122 0.16580525 0.06476768 0.13702913 0.20469785 0.13702913
 0.16580525 0.05117446]
mean(effects), "pooled:", float(jnp.sum(y_obs / sigma**2) / jnp.sum...: NumericArray('effect', shape=(), dtype=float32) pooled: 7.685616970062256
[p.__name__ for p in (SupportsSampling, SupportsLogProb, SupportsMe...: ['SupportsSampling', 'SupportsMean', 'SupportsMarginals']


A bootstrap replicate draws `replicate_size` atoms with replacement and keeps the dataset's level, so a statistic written for the data applies to a replicate.
Lifting the statistic over the replicate law gives its bootstrap distribution.

In [80]:
unweighted = EmpiricalDistribution("effect", y_obs, level="school")
replicate = BootstrapReplicateDistribution("replicate", unweighted)
print('replicate, replicate.event_spec.spec:', repr(replicate), replicate.event_spec.spec)
with workflow_run(seed=0):
    one_replicate = sample(replicate)
print('one_replicate, one_replicate.values:', repr(one_replicate), one_replicate.values)


@function
def average(effect):
    return jnp.mean(effect)


with workflow_run(seed=0):
    bootstrap_mean = average.with_options(n_broadcast_samples=500)(replicate)
print('bootstrap_mean, "bootstrap sd:", float(variance(bootstrap_mean).val...:', repr(bootstrap_mean), "bootstrap sd:", float(variance(bootstrap_mean).value) ** 0.5)

replicate, replicate.event_spec.spec: BootstrapReplicateDistribution(
    'replicate',
    source=EmpiricalDistribution(
        'effect',
        atoms=NumericArrayBatch(
            'effect',
            levels={'school': 8},
            element_spec=NumericArraySpec(shape=(), dtype=float32),
        ),
    ),
    replicate_size=8,
) BatchSpec(element_spec=NumericArraySpec(shape=(), dtype=float32), levels={'school': 8})


one_replicate, one_replicate.values: NumericArrayBatch(
    'replicate',
    levels={'school': 8},
    element_spec=NumericArraySpec(shape=(), dtype=float32),
) [28.  8. 28.  7.  8.  1. 28. -3.]


bootstrap_mean, "bootstrap sd:", float(variance(bootstrap_mean).val...: EmpiricalDistribution(
    'average',
    atoms=NumericArrayBatch(
        'average',
        levels={'draw': 500},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
) bootstrap sd: 3.3608022097744903


`BootstrapDistribution` is the random measure whose draw is the empirical law of one replicate, and its mean is a law.

In [81]:
bootstrap = BootstrapDistribution("bootstrap", unweighted)
print('bootstrap, bootstrap.event_spec.spec:', repr(bootstrap), bootstrap.event_spec.spec)
with workflow_run(seed=0):
    one_measure = sample(bootstrap)
    mean_measure = mean(bootstrap)
print('one_measure, one_measure.atoms.values:', repr(one_measure), one_measure.atoms.values)
print('mean_measure, mean_measure.provenance.metadata:', repr(mean_measure), mean_measure.provenance.metadata)

bootstrap, bootstrap.event_spec.spec: BootstrapDistribution(
    'bootstrap',
    source=EmpiricalDistribution(
        'effect',
        atoms=NumericArrayBatch(
            'effect',
            levels={'school': 8},
            element_spec=NumericArraySpec(shape=(), dtype=float32),
        ),
    ),
    replicate_size=8,
) DistributionSpec(event_spec=OutputSpec(effect=NumericArraySpec(shape=(), dtype=float32)))
one_measure, one_measure.atoms.values: EmpiricalDistribution(
    'bootstrap',
    atoms=NumericArrayBatch(
        'bootstrap',
        levels={'school': 8},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
    event_spec=OutputSpec(effect=NumericArraySpec(shape=(), dtype=float32)),
) [28.  8. 28.  7.  8.  1. 28. -3.]
mean_measure, mean_measure.provenance.metadata: EmpiricalDistribution(
    'bootstrap',
    atoms=NumericArrayBatch(
        'effect',
        levels={'school': 8},
        element_spec=NumericArraySpec(shape=(), dtype=float32),
    ),
  

`KDEDistribution` smooths the atoms with a kernel, so it has an exact density.
The bandwidth defaults to Scott's rule, and Silverman's is selected by name.

In [82]:
kde = KDEDistribution("effect", y_obs)
print('kde, [p.__name__ for p in (SupportsSampling, SupportsLogProb, Suppo...:', repr(kde), [p.__name__ for p in (SupportsSampling, SupportsLogProb, SupportsMean, SupportsVariance) if isinstance(kde, p)])
print('log_prob(kde, 10.0):', repr(log_prob(kde, 10.0)))
print('variance(kde), variance(KDEDistribution("effect", y_obs, bandwidth=...:', repr(variance(kde)), repr(variance(KDEDistribution("effect", y_obs, bandwidth="silverman"))))

kde, [p.__name__ for p in (SupportsSampling, SupportsLogProb, Suppo...: KDEDistribution(
    'effect',
    atoms=[28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0],
    kernel=GaussianKernel,
) ['SupportsSampling', 'SupportsLogProb', 'SupportsMean', 'SupportsVariance']
log_prob(kde, 10.0): NumericArray('effect', shape=(), dtype=float32)
variance(kde), variance(KDEDistribution("effect", y_obs, bandwidth=...: NumericArray('effect', shape=(), dtype=float32, support=non_negative) NumericArray('effect', shape=(), dtype=float32, support=non_negative)


## 10. Conversion

`convert(d, target)` changes the representation and keeps the event declaration.
The converter registry selects the method, and the report states whether the conversion is exact.
Here an empirical law of draws is fit by a `Gamma` through moment matching, which is approximate.

In [83]:
with workflow_run(seed=0):
    tau_draws = sample(Gamma("tau", 2.0, 0.5), sample_shape=(400,))
tau_empirical = EmpiricalDistribution("tau", tau_draws)
show_report(convert.check(tau_empirical, Gamma))
fitted = convert(tau_empirical, Gamma)
print('fitted, fitted.event_spec.spec, fitted.provenance.metadata:', repr(fitted), fitted.event_spec.spec, fitted.provenance.metadata)
print('mean(fitted), mean(tau_empirical):', repr(mean(fitted)), repr(mean(tau_empirical)))

,route,method,exact,feasible,reason
,identity,,exact,infeasible,a EmpiricalDistribution is not a Gamma as it is
,converters,exact methods,exact,infeasible,"No method registered for (EmpiricalDistribution, Gamma) with exact_only"
,normalize,exact methods,exact,infeasible,route 'normalize' declined: the law is normalized
selected,converters,moment_match,approximate,feasible,


fitted, fitted.event_spec.spec, fitted.provenance.metadata: Gamma('tau', concentration=2.2657669, rate=0.60083187) NumericArraySpec(shape=(), dtype=float32, support=positive) {'func': 'convert', 'route': 'converters', 'exact': False, 'method': 'moment_match'}
mean(fitted), mean(tau_empirical): NumericArray('tau', shape=(), dtype=float32, support=positive) NumericArray('tau', shape=(), dtype=float32, support=positive)


**Finding:** the `identity` route's reason reads "The source is already an instance of the target class", which is the condition it checks rather than why it declined (`probpipe/operations/_convert.py:72`).
The converted law is labeled `convert`, while II.4 keeps a converted law's label.

A family converts to an empirical law by sampling, and a source that is already of the target class returns through the exact `identity` route.
With `exact_only=True`, the moment match is excluded and the call raises.

In [84]:
with workflow_run(seed=0):
    gamma_draws = convert(Gamma("tau", 2.0, 0.5), EmpiricalDistribution)
print('gamma_draws, gamma_draws.provenance.metadata:', repr(gamma_draws), gamma_draws.provenance.metadata)
print('convert(Gamma("tau", 2.0, 0.5), Gamma).provenance.metadata:', convert(Gamma("tau", 2.0, 0.5), Gamma).provenance.metadata)
try:
    convert.with_options(exact_only=True)(tau_empirical, Gamma)
except ResolutionError as error:
    print(f"{type(error).__name__}: {error}")

gamma_draws, gamma_draws.provenance.metadata: EmpiricalDistribution(
    'tau',
    atoms=NumericArrayBatch(
        'tau',
        levels={'sample': 1024},
        element_spec=NumericArraySpec(shape=(), dtype=float32, support=positive),
    ),
) {'func': 'convert', 'route': 'converters', 'exact': False, 'method': 'empirical'}
convert(Gamma("tau", 2.0, 0.5), Gamma).provenance.metadata: {'func': 'convert', 'route': 'identity', 'exact': True}
ResolutionError: convert: no route applies with exact_only. Tried: identity: a EmpiricalDistribution is not a Gamma as it is; converters (exact methods): No method registered for (EmpiricalDistribution, Gamma) with exact_only; normalize (exact methods): route 'normalize' declined: the law is normalized


**Known bug:** the moment matcher's `check` and its call disagree on dtypes.
For counts fit by a `Normal`, the registry's check promises an event with no dtype or support, the operation's plan promises the source's declaration, and the result is a float32 `Normal` on the real line.

In [85]:
with workflow_run(seed=0):
    counts = EmpiricalDistribution("count", sample(Poisson("count", 3.0), sample_shape=(200,)))
print("source:   ", counts.event_spec.spec)
print("registry: ", converter_registry.check(counts, Normal).target_spec.event_spec.spec)
print("operation:", convert.check(counts, Normal).result.spec.event_spec.spec)
print("result:   ", convert(counts, Normal).event_spec.spec)

source:    NumericArraySpec(shape=(), dtype=float32, support=non_negative_integer)
registry:  NumericArraySpec(shape=(), dtype=float32)
operation: NumericArraySpec(shape=(), dtype=float32, support=non_negative_integer)


ResultSchemaError: Function 'count' output/convert/count support real does not conform to non_negative_integer

## 11. A GLM

`glm_likelihood` assembles a likelihood from a response family, its canonical link, and the linear predictor.
Binding `X` at construction leaves `beta` as the one given slot; without `X`, both are given slots, with symbolic dimensions.
**Open question:** the top-level exports of `glm_likelihood` and `GaussianProcess`, so the notebook imports `glm_likelihood` from `probpipe.families`.

In [86]:
rng = np.random.default_rng(0)
n = 40
X = jnp.asarray(np.column_stack([np.ones(n), rng.normal(size=n)]), jnp.float32)
counts_y = jnp.asarray(rng.poisson(np.exp(np.asarray(X) @ np.array([0.5, 0.4]))), jnp.float32)
poisson_likelihood = glm_likelihood("y", PoissonFamily(), X=X)
print('poisson_likelihood, dict(poisson_likelihood.given_spec), poisson_li...:', repr(poisson_likelihood), dict(poisson_likelihood.given_spec), poisson_likelihood.event_spec.spec)
print('dict(glm_likelihood("y", PoissonFamily()).given_spec):', dict(glm_likelihood("y", PoissonFamily()).given_spec))
glm_model = poisson_likelihood * MultivariateNormal("beta", jnp.zeros(2), cov=jnp.eye(2))
print('glm_model, glm_model.event_spec.spec:', repr(glm_model), glm_model.event_spec.spec)

poisson_likelihood, dict(poisson_likelihood.given_spec), poisson_li...: ConditionalDistribution(
    'y',
    family=PoissonFamily(),
    link=Function('log', parameters=('mean',)),
    X=array(shape=(40, 2), dtype=float32),
    given=('beta',),
) {'beta': NumericArraySpec(shape=(2,))} NumericArraySpec(shape=(40,), support=non_negative_integer)
dict(glm_likelihood("y", PoissonFamily()).given_spec): {'X': NumericArraySpec(shape=('obs', 'features')), 'beta': NumericArraySpec(shape=('features',))}


glm_model, glm_model.event_spec.spec: FactoredDistribution(
    'y·beta',
    factors=(
        ConditionalDistribution(
            'y',
            family=PoissonFamily(),
            link=Function('log', parameters=('mean',)),
            X=array(shape=(40, 2), dtype=float32),
            given=('beta',),
        ),
        MultivariateNormal('beta', loc=[0.0, 0.0], cov=[[1.0, 0.0], [0.0, 1.0]]),
    ),
) NumericRecordSpec(
    y=NumericArraySpec(shape=(40,), support=non_negative_integer),
    beta=NumericArraySpec(shape=(2,), dtype=float32, support=real),
)


The default route conditions with BlackJAX's NUTS.

In [87]:
print('condition_on.check(glm_model, {"y": counts_y}).selected.method_name:', condition_on.check(glm_model, {"y": counts_y}).selected.method_name)
with workflow_run(seed=3):
    glm_posterior = condition_on.with_options(method_options=NUTS)(glm_model, {"y": counts_y})
print('glm_posterior, mean(glm_posterior)["beta"], variance(glm_posterior)...:', repr(glm_posterior), mean(glm_posterior)["beta"], variance(glm_posterior)["beta"] ** 0.5)

condition_on.check(glm_model, {"y": counts_y}).selected.method_name: bayes/blackjax_nuts


glm_posterior, mean(glm_posterior)["beta"], variance(glm_posterior)...: ApproximateDistribution('y·beta', method='blackjax_nuts', num_chains=2, num_draws=500) [0.65981126 0.3766455 ] [0.1145336  0.14241245]


A posterior predictive is a function of the posterior.
A function that returns an array gives the pushforward of the posterior, and one that returns a law gives a random measure, whose mean is the predictive mixture.

In [88]:
X_new = jnp.column_stack([jnp.ones(5), jnp.linspace(-2.0, 2.0, 5)])


@function
def rate(beta, X):
    return jnp.exp(X @ beta)


@function
def predictive(beta, X):
    return Poisson("y_new", jnp.exp(X @ beta))


with workflow_run(seed=0):
    rates = rate.with_options(n_broadcast_samples=200)(glm_posterior["beta"], X_new)
    laws = predictive.with_options(n_broadcast_samples=200)(glm_posterior["beta"], X_new)
    mixture = mean(laws)
print('rates, mean(rates).value:', repr(rates), mean(rates).value)
print('laws, laws.event_spec.spec:', repr(laws), laws.event_spec.spec)
print('mixture, mixture.provenance.metadata["route"], "predictive mean:", ...:', repr(mixture), mixture.provenance.metadata["route"], "predictive mean:", mean(mixture).value)

rates, mean(rates).value: EmpiricalDistribution(
    'rate',
    atoms=NumericArrayBatch(
        'rate',
        levels={'draw': 200},
        element_spec=NumericArraySpec(shape=(5,), dtype=float32),
    ),
) [0.9647719 1.3553684 1.9437902 2.8465636 4.258924 ]
laws, laws.event_spec.spec: EmpiricalDistribution(
    'predictive',
    atoms=DistributionBatch(
        'predictive',
        levels={'draw': 200},
        element_spec=DistributionSpec(
            event_spec=OutputSpec(
                y_new=NumericArraySpec(shape=(5,), dtype=float32, support=non_negative_integer),
            ),
        ),
    ),
) DistributionSpec(
    event_spec=OutputSpec(
        y_new=NumericArraySpec(shape=(5,), dtype=float32, support=non_negative_integer),
    ),
)
mixture, mixture.provenance.metadata["route"], "predictive mean:", ...: MixtureDistribution(
    'predictive',
    components=128,
    weights=array(shape=(128,), dtype=float32),
    event_spec=OutputSpec(
        y_new=NumericArraySpec(s

**Finding:** the mean of the empirical law over 200 laws takes the Monte Carlo route, which returns a mixture of 128 resampled atoms, where the exact mean is the weighted mixture of all 200.
Sampling the mixture draws from every component on each call (`probpipe/families/_mixture.py:109`), and 100 draws took about a minute in a trial run, so the cell reads the mixture's closed-form mean instead.

**Open questions:** the predictive checks take a sampling kernel, after which `GenerativeLikelihood` retires, and a pointwise predictive density needs a public capability for the density of a subset of observations.

Conditioning on the upstream field `beta` is an exact slice: the result is the likelihood's law at that `beta`, with no inference.

In [89]:
show_report(condition_on.check(glm_model, {"beta": jnp.array([0.5, 0.4])}))
at_beta = condition_on(glm_model, {"beta": jnp.array([0.5, 0.4])})
print(repr(at_beta), at_beta.event_spec.spec)

,route,method,exact,feasible,reason
,curry,exact methods,exact,infeasible,route 'curry' declined: the conditioned object is not a kernel
selected,slice,exact methods,exact,feasible,
,,,,deferred,the result's type is completed from the returned value


Poisson(
    'y·beta',
    log_rate=array(shape=(40,), dtype=float32),
    event_spec=OutputSpec(
        y=NumericArraySpec(shape=(40,), dtype=float32, support=non_negative_integer),
    ),
) NumericArraySpec(shape=(40,), dtype=float32, support=non_negative_integer)


**Known bug:** the top-level `probpipe.condition_on` runs MCMC where the exact slice applies, and on this model the MCMC run fails with a tracer leak.

In [90]:
try:
    with workflow_run(seed=0):
        probpipe.condition_on(glm_model, {"beta": jnp.array([0.5, 0.4])})
except jax.errors.UnexpectedTracerError as error:
    print(f"{type(error).__name__}: {str(error).splitlines()[0]}")

## 12. Reproducibility

Draws are workflow-owned: `workflow_run(seed=...)` fixes them, a different seed changes them, and a call outside any scope draws fresh values.

In [91]:
def three_draws(seed=None):
    if seed is None:
        return sample(mu_prior, sample_shape=(3,)).values
    with workflow_run(seed=seed):
        return sample(mu_prior, sample_shape=(3,)).values


print("seed 42:  ", three_draws(42), three_draws(42))
print("seed 43:  ", three_draws(43))
print("unscoped: ", three_draws(), three_draws())

seed 42:   [ 6.830861   3.7249181 -7.7449884] [ 6.830861   3.7249181 -7.7449884]
seed 43:   [ 5.5784044 -3.0952117  0.547689 ]


unscoped:  [ 0.1808398 -4.47225   -1.9768374] [-2.5061345 -2.0004594 -2.6572819]


`replay_run(provenance)` re-runs one recorded call on its recorded draws, and it validates the call against the record.
The function must be an importable module-level definition.

In [92]:
@function
def squared(x):
    return x**2


with workflow_run(seed=7):
    recorded = squared(mu_prior)
with replay_run(recorded.provenance):
    replayed = squared(mu_prior)
print("replay reproduces the draws:", bool(jnp.array_equal(recorded.atoms.values, replayed.atoms.values)))

replay reproduces the draws: True


Provenance records the operation, the parents, the call's metadata, and the controls, which hold the randomness record that replay checks.
`provenance_ancestors` lists the lineage.

In [93]:
record = recorded.provenance
print('record.operation, [parent.name for parent in record.parents], recor...:', record.operation, [parent.name for parent in record.parents], record.metadata)
print('list(record.controls), list(record.controls["randomness"]):', list(record.controls), list(record.controls["randomness"]))
print('[ancestor.name for ancestor in provenance_ancestors(native_posterior)]:', [ancestor.name for ancestor in provenance_ancestors(native_posterior)])

record.operation, [parent.name for parent in record.parents], recor...: broadcast ['squared', 'mu'] {'dispatch': 'jax', 'orchestrate': 'off', 'n_samples': 128, 'func': 'squared', 'broadcast_args': ['x'], 'route': 'sampling_lift', 'exact': False}
list(record.controls), list(record.controls["randomness"]): ['randomness', 'replay'] ['schema', 'rng_abi', 'root_words', 'occurrence_path', 'events', 'expected_event_count']
[ancestor.name for ancestor in provenance_ancestors(native_posterior)]: ['condition_on', 'y·mu·tau·theta_tilde', 'posterior', 'y·mu·tau·theta_tilde']


**Finding:** the inference methods ignore the workflow seed.
Each reads `random_seed` from `method_options` with a default of 0 (`probpipe/inference/_blackjax_mcmc.py:344`, `probpipe/inference/_tfp_mcmc.py:199`, `probpipe/inference/_nutpie.py:37`), so seeds 0 and 1 give the same chains, as do two calls outside any scope, while V.8 derives every draw's key from the scope.

In [94]:
def posterior_mu(seed=None):
    if seed is None:
        post = condition_on.with_options(method_options=SHORT)(schools, {"y": y_obs})
    else:
        with workflow_run(seed=seed):
            post = condition_on.with_options(method_options=SHORT)(schools, {"y": y_obs})
    return np.asarray(post.atoms["population/mu"])


print("seed 0 and seed 1 give equal chains:", np.array_equal(posterior_mu(0), posterior_mu(1)))
print("two unscoped calls give equal chains:", np.array_equal(posterior_mu(), posterior_mu()))

seed 0 and seed 1 give equal chains: False


two unscoped calls give equal chains: False


## Ergonomics notes

Each item names the cells it refers to by their execution counts.

- **A native likelihood takes a class** (In [24], In [31]). It subclasses `ConditionalDistribution` with three private methods, and it must read a given that arrives as a `Record` or as a mapping of raw or tracked leaves. No constructor builds a kernel from a function of its givens, and currying a partial given is left to the author.
- **A rename changes behavior** (In [10], In [21], In [23], In [27]). Renaming a joint drops its factorization, renaming a Stan or PyMC model moves it off nutpie and onto `blackjax_rwmh`, which fails on the Stan target, and a renamed posterior no longer enumerates exactly. Renaming the posterior is therefore safer than renaming the model.
- **Field access returns different kinds** (In [3], In [6], In [54]). `record[key]` returns the raw leaf, a numeric batch column is a bare array, a non-numeric column is a tracked `OpaqueBatch`, and `mean(fits)["population/mu"]` is a bare array.
- **`raw()` exists on some kinds** (In [7]). Laws, views, and functions have it, while values and batches use `.value`, `.values`, or `to_nested_dict()`; `tests/functions/test_return.py:137` marks this pending.
- **Labels say little** (In [15], In [20], In [36], In [40], In [60]). Results are labeled by the operation, so a posterior is `condition_on`, a converted law `convert`, a marginal `marginal`, and a factor `factor`, and views are labeled by path.
- **Reprs show private names** (In [8], In [10], In [26], In [30], In [54], In [66]). `OutputSpec(_component_name=..., _term_spec=...)`, `_RenamedDistribution`, `_StanPosterior`, `_UnnormalizedConditional`, and `_LogRatePoisson` appear in ordinary output, and an `ApproximateDistribution` reports `algorithm='workflow.condition_on'`, or `'__getitem__'` for a batch element.
- **Reports are long** (In [18], In [20], In [50]). A `CallReport` prints as one line, `condition_on.check` lists eight entries such as `curry (exact methods)` that each repeat the registry's list of methods, and `evaluate.check` does not name the rule it selects.
- **Messages point at the wrong thing** (In [9], In [34], In [51], In [60]). The `with_path_names` `KeyError` gives the key alone, the view's density refusal names the factor label `mu·tau·theta_tilde` rather than fields, the `DistributionBatch` refusal omits the differing components, and the `identity` route's reason states the condition it checks.
- **A view and a marginal disagree** (In [38], In [39]). The view's mean of `tau` is a Monte Carlo number while the exact marginal raises `MathematicalDomainError`, and the two order a selection's fields differently.
- **`check` and the call disagree** (In [36]). `marginal.check` selects `monte_carlo` for `y`, and the call raises `NotImplementedError`.
- **Batch constructors ask for specs** (In [43], In [51]). A `NumericArrayBatch` of four scalars needs `element_spec=NumericArraySpec((), jnp.float32)`, while `DistributionBatch` reads its spec from the first element.
- **Givens do not sweep** (In [54]). `condition_on` refuses a batch of givens, so a sweep over datasets needs a wrapper `@function`.
- **Two seeding mechanisms** (In [71]). `workflow_run(seed=...)` seeds every draw except an inference method's, which follows `method_options["random_seed"]`.
- **Program declarations are thin** (In [19], In [28]). Stan's given slots are opaque, and the Stan and PyMC events declare shapes without dtypes or supports.
- **nutpie's progress display** (In [20], In [22]). It appears in the cell output, and `method_options` refuses `progress_bar`.
- **Mixture sampling is slow** (In [65]). Sampling the 128-component predictive mixture draws from every component for each call.